In [38]:
import os
from dotenv import load_dotenv
import pypdf

In [17]:
import sys
!{sys.executable} -m pip install pypdf

In [25]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [26]:
# [PASO 1] Cargar la API Key
load_dotenv()

True

In [27]:
# [PASO 2] Configurar el Traductor Matemático (Embeddings)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [28]:
# [PASO 3] LEER LOS ARCHIVOS DE LA CARPETA 'DOCS'
carpeta_documentos = "docs"

In [29]:
# CLAVE DE LA SOLUCIÓN: Creamos la lista vacía aquí para que Python sepa qué es
fragmentos_finales = []

In [30]:
print("=== [1] Leyendo archivos de la carpeta 'docs' ===")

=== [1] Leyendo archivos de la carpeta 'docs' ===


In [31]:
if not os.path.exists(carpeta_documentos):
    print(f"❌ ERROR: No encuentro la carpeta '{carpeta_documentos}'")
else:
    for nombre_archivo in os.listdir(carpeta_documentos):
        if nombre_archivo.endswith(".pdf"):
            ruta_completa = os.path.join(carpeta_documentos, nombre_archivo)
            print(f"📄 Procesando archivo: {nombre_archivo}")
            
            # Abrimos el PDF físico
            lector_pdf = pypdf.PdfReader(ruta_completa)
            
            # Recorremos cada página del documento
            for indice_pag, pagina in enumerate(lector_pdf.pages):
                texto_completo_pagina = pagina.extract_text()
                
                # Configuración de nuestro fraccionador manual
                tamano_bloque = 500  # Máximo 500 caracteres por pedazo
                solapamiento = 50   # 50 caracteres de carrerilla para el contexto
                
                inicio = 0
                while inicio < len(texto_completo_pagina):
                    fin = inicio + tamano_bloque
                    trozo_texto = texto_completo_pagina[inicio:fin]
                    
                    # Empacamos el fragmento como un Documento de LangChain
                    objeto_documento = Document(
                        page_content=trozo_texto,
                        metadata={
                            "fuente": nombre_archivo, 
                            "pagina": indice_pag + 1
                        }
                    )
                    
                    # Ahora sí, guardamos el fragmento en la lista existente
                    fragmentos_finales.append(objeto_documento)
                    
                    # Avanzamos el puntero calculando el solapamiento
                    inicio += (tamano_bloque - solapamiento)

print(f"\n✂️ ¡Éxito total! Se crearon {len(fragmentos_finales)} fragmentos usando Python puro.")

📄 Procesando archivo: datawithia_faqs.pdf
📄 Procesando archivo: datawithia_politicas.pdf

✂️ ¡Éxito total! Se crearon 10 fragmentos usando Python puro.


In [32]:
# [PASO 4] Guardar en el Archivador Inteligente (RAM)
vectorstore = InMemoryVectorStore.from_documents(fragmentos_finales, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [33]:
# [PASO 5] Diseñar el Prompt del Asistente Corporativo
template = """Actúa como un asistente de soporte experto para Datawithia. 
Responde la consulta del usuario basándote únicamente en el contexto provisto.
Si no sabes la respuesta o si no está en los documentos, di amablemente que no posees esa información.

Contexto de la empresa:
{context}

Pregunta del usuario: {question}
Respuesta Institucional:"""

In [34]:
prompt = ChatPromptTemplate.from_template(template)
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.1)

In [35]:
# [PASO 6] Construir la Línea de Ensamblaje LCEL
chain = (
    {
        "context": retriever, 
        "question": RunnablePassthrough()
    }
    | prompt 
    | model 
    | StrOutputParser()
)

In [36]:
# [PASO 7] Probando el sistema con preguntas reales
print("\n=== [2] Sistema RAG Listo. Iniciando Consultas ===")


=== [2] Sistema RAG Listo. Iniciando Consultas ===


In [ ]:
# [MEJORA 2] Helper con manejo del error de cupo (429 RESOURCE_EXHAUSTED).
# El plan gratuito de Gemini tiene un cupo diario de peticiones. Si se agota,
# la API lanza un 429. En vez de que el script "explote" con un traceback feo,
# lo capturamos y mostramos un mensaje claro para seguir estudiando con calma.
def preguntar_al_rag(pregunta):
    print(f"\n🙋‍♂️ Consulta: {pregunta}")
    try:
        respuesta = chain.invoke(pregunta)
        print(f"🤖 Gemini dice: {respuesta}")
    except Exception as error:
        # Detectamos el caso típico de cupo agotado para dar una pista útil.
        if "RESOURCE_EXHAUSTED" in str(error) or "429" in str(error):
            print("⏳ Cupo de la API agotado (429). Espera unos segundos/minutos "
                  "o revisa tu límite diario en https://ai.dev/rate-limit")
        else:
            print(f"❌ Error inesperado al consultar: {error}")

# === [PASO 7] Ejecución de tus consultas reales ===
print("\n=== [2] Iniciando Consultas al RAG ===")

preguntar_al_rag("¿Quién mantiene el control de las llaves de API y credenciales del cliente?")
preguntar_al_rag("¿En qué momento los flujos, scripts y configuraciones personalizadas pasan a ser propiedad exclusiva del cliente?")


🙋‍♂️ Consulta: ¿Quién mantiene el control de las llaves de API y credenciales del cliente?
🤖 Gemini dice: El cliente mantiene siempre el control de sus llaves de API y credenciales.
